In [ ]:
!pip -q install "librosa>=0.10.2" soundfile

In [ ]:
import os
import urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import librosa.display
import matplotlib.pyplot as plt

from IPython.display import Audio, display
from google.colab import files

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
MODEL_PATH = "/content/best_cnn2d_mel.pt"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "Không tìm thấy /content/best_cnn2d_mel.pt. "
        "Hãy mở Files ở thanh bên trái của Colab và upload checkpoint trước."
    )

print("✓ Model found:", MODEL_PATH)


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )

    def forward(self, x):
        return self.block(x)

class CNN2D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32, 0.05),
            ConvBlock(32, 64, 0.10),
            ConvBlock(64, 128, 0.15),
            ConvBlock(128, 192, 0.20),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)

CLASSES = checkpoint.get(
    "classes",
    ["cailuong", "catru", "chauvan", "cheo", "hatxam"]
)

model = CNN2D(num_classes=len(CLASSES))
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model = model.to(DEVICE)
model.eval()

print("Checkpoint loaded successfully.")
print("Seed:", checkpoint.get("seed"))
print("Epoch:", checkpoint.get("epoch"))
print("Representation:", checkpoint.get("representation"))
print("Classes:", CLASSES)

In [ ]:
MEL = checkpoint.get("mel_config", {
    "sr": 22050,
    "n_fft": 2048,
    "hop_length": 512,
    "n_mels": 128,
    "fmin": 0,
    "fmax": 11025.0,
    "top_db": 80,
})

TARGET_SECONDS = 30
TARGET_H = 256
TARGET_W = 320

def preprocess_audio(audio_path):
    sr = int(MEL["sr"])
    y, _ = librosa.load(audio_path, sr=sr, mono=True)

    target_len = sr * TARGET_SECONDS
    original_duration = len(y) / sr

    if len(y) >= target_len:
        y30 = y[:target_len]
        action = "cropped to first 30 seconds" if len(y) > target_len else "already 30 seconds"
    else:
        y30 = np.pad(y, (0, target_len - len(y)))
        action = "zero-padded to 30 seconds"

    mel_power = librosa.feature.melspectrogram(
        y=y30,
        sr=sr,
        n_fft=int(MEL["n_fft"]),
        hop_length=int(MEL["hop_length"]),
        n_mels=int(MEL["n_mels"]),
        fmin=float(MEL["fmin"]),
        fmax=float(MEL["fmax"]),
        power=2.0,
    )

    mel_db = librosa.power_to_db(
        mel_power,
        ref=np.max,
        top_db=float(MEL["top_db"])
    )

    
    x = (mel_db + float(MEL["top_db"])) / float(MEL["top_db"])
    x = np.clip(x, 0.0, 1.0).astype(np.float32)

    
    tensor = torch.from_numpy(x)[None, None, ...]
    tensor = F.interpolate(
        tensor,
        size=(TARGET_H, TARGET_W),
        mode="bilinear",
        align_corners=False
    )[0]

    return y, y30, sr, mel_db, tensor, original_duration, action

In [ ]:
print("Upload một file WAV / MP3 / FLAC / M4A / OGG:")
uploaded_audio = files.upload()

if not uploaded_audio:
    raise RuntimeError("Chưa upload audio.")

AUDIO_PATH = next(iter(uploaded_audio))
print("Selected:", AUDIO_PATH)

display(Audio(AUDIO_PATH))

In [ ]:
y, y30, sr, mel_db, x, duration, action = preprocess_audio(AUDIO_PATH)

with torch.inference_mode():
    logits = model(x.unsqueeze(0).to(DEVICE))
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

pred_idx = int(np.argmax(probs))
pred_class = CLASSES[pred_idx]
confidence = float(probs[pred_idx])

DISPLAY_NAMES = {
    "cailuong": "Cải lương",
    "catru": "Ca trù",
    "chauvan": "Chầu văn",
    "cheo": "Chèo",
    "hatxam": "Hát xẩm",
}

print("=" * 55)
print("PREDICTION")
print("=" * 55)
print(f"File              : {AUDIO_PATH}")
print(f"Original duration : {duration:.2f} s")
print(f"Preprocessing     : {action}")
print(f"Sample rate       : {sr} Hz")
print(f"Predicted genre   : {DISPLAY_NAMES.get(pred_class, pred_class)}")
print(f"Confidence        : {confidence * 100:.2f}%")
print("=" * 55)

order = np.argsort(probs)[::-1]
for rank, idx in enumerate(order, 1):
    print(f"{rank}. {DISPLAY_NAMES.get(CLASSES[idx], CLASSES[idx]):12s} {probs[idx]*100:6.2f}%")

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.specshow(
    mel_db,
    sr=sr,
    hop_length=int(MEL["hop_length"]),
    x_axis="time",
    y_axis="mel",
    fmax=float(MEL["fmax"])
)
plt.colorbar(format="%+2.0f dB")
plt.title("Log-Mel Spectrogram — 30-second model input")
plt.tight_layout()
plt.show()

names = [DISPLAY_NAMES.get(c, c) for c in CLASSES]

plt.figure(figsize=(9, 4))
bars = plt.bar(names, probs * 100)
plt.ylabel("Probability (%)")
plt.ylim(0, 100)
plt.title(
    f"Prediction: {DISPLAY_NAMES.get(pred_class, pred_class)} "
    f"({confidence*100:.2f}%)"
)
for bar, p in zip(bars, probs):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        min(p*100 + 1.5, 97),
        f"{p*100:.1f}%",
        ha="center",
        va="bottom"
    )
plt.tight_layout()
plt.show()